In [15]:
# в этой части мы анализируем данные из гугл таблички с ценами на яйца четырех крупных производителей яиц


In [16]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import math
import plotly.graph_objects as go
# импорт библиотек

In [17]:
from google.colab import files
uploaded = files.upload()
# загрузка файла с ключами апи

Saving credseggo44.json to credseggo44 (1).json


In [22]:
scopes = ["https://www.googleapis.com/auth/spreadsheets"]
# аналогично парсингу ответов гугл формы

# Подключение к API по ключам
creds = Credentials.from_service_account_file('credseggo44.json', scopes=scopes)
client = gspread.authorize(creds)

# Открываем таблицу id
sheet_id = "1ZtPFyBSvnYSUUQHEFl07YTf8pGSRK2AgLHMn_IH5FRA"
sheet = client.open_by_key(sheet_id)

companies = ["Село Зеленое", "Доброе Подворье", "Окское", "Молодецкие"]
means_list = []
elasticity_list = []
std_list = []

In [23]:
import plotly.express as px
for company in companies:
    worksheet = sheet.worksheet(company)
    data = worksheet.get_all_values()
    df = pd.DataFrame(data[1:], columns=data[0])
    for col in df.columns[1:5]:
        df[col] = df[col].str.replace(',', '.').astype(float)

    # графики построены не без помощи нейросетей
    fig = px.line(
        df,
        x="Дата",
        y="Средняя цена, ₽",
        title=f'График цены яиц бренда "{company}"'
    )

    # Настройка оси X: метки каждые 30 значений и поворот меток на 90 градусов
    fig.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=df["Дата"][::30],
            tickangle=270,
            title_font=dict(size=40)  # Увеличение шрифта названия оси X
        ),
        yaxis=dict(
            title_font=dict(size=40)  # Увеличение шрифта названия оси Y
        ),
        title=dict(
            font=dict(size=60)  # Увеличение шрифта заголовка графика
        ),
        width=1600,
        height=900
    )

    fig.show()

    # я участвовал в кейс чемпионате и писал функцию для нахождения эластичности спроса по цене. применим ее и здесь
    # Функция для удаления выбросов
    def remove_outliers(df, column):
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        # Фильтр для оставления данных в пределах 1.5 * IQR от квартилей
        filter = (df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)
        return df[filter]

    clean_data = df.copy()
    for col in ['Средняя цена, ₽', 'Индекс продаж (шт.), k']:
        clean_data = remove_outliers(clean_data, col)

    # Берем логарифмы для подсчета эластичности
    clean_data['log_Price'] = np.log(clean_data['Средняя цена, ₽'])
    clean_data['log_Quantity'] = np.log(clean_data['Индекс продаж (шт.), k'])
    # Построение регрессии: log(Quantity) ~ log(Price)
    model = smf.ols("log_Quantity ~ log_Price", data=clean_data).fit()

    elasticity = model.params["log_Price"]
    a = math.e ** model.params["Intercept"]
    mn = np.mean(df["Средняя цена, ₽"])
    print(mn)
    std = np.std(df["Средняя цена, ₽"])
    std_list.append(std)
    means_list.append(mn)
    elasticity_list.append(elasticity)



114.36568699478882


103.05700104536061


166.12793091437945


225.8007378487013


In [24]:

new_df = pd.DataFrame({"Company": companies, "Mean value": means_list, "Standard Deviation": std_list, "Demand Elasticity": elasticity_list})
new_df.head()

[np.float64(114.36568699478882), np.float64(103.05700104536061), np.float64(166.12793091437945), np.float64(225.8007378487013)]


,Company,Mean value,Standard Deviation,Demand Elasticity
0,Село Зеленое,114.365687,20.653833,-0.024417
1,Доброе Подворье,103.057001,20.957512,1.840502
2,Окское,166.127931,19.764922,-2.568976
3,Молодецкие,225.800738,48.048690,0.265041


In [25]:
# табличка построена не без помощи нейросетей
fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=list(new_df.columns),
                fill_color='lightgrey',
                align='center',
                font=dict(color='black', size=14),
                height=30
            ),
            cells=dict(
                values=[np.round(new_df[col], decimals=3) for col in new_df.columns],
                fill_color=['lavender', 'white'],
                align='center',
                font=dict(color='black', size=12),
                height=25
            )
        )
    ]
)

# Настройка размеров и названия
fig.update_layout(
    title='Данные о яичках крупных производителей',
    margin=dict(l=5, r=5, t=40, b=5),
    width=500,
    height=300
)

# Показ таблицы
fig.show()
